**Simulated Dataset for Game-Based Learning Quiz**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw")
RAW_FILE = os.path.join(RAW_PATH, "quiz_interactions.csv")

PROJECT_ROOT, RAW_FILE


('/content/drive/MyDrive/bigdata-gbl-quiz',
 '/content/drive/MyDrive/bigdata-gbl-quiz/data/raw/quiz_interactions.csv')

In [ ]:
os.path.exists(RAW_PATH), RAW_PATH


(True, '/content/drive/MyDrive/bigdata-gbl-quiz/data/raw')

**Generate realistic simulated data (core code)**

In [ ]:
import numpy as np
import pandas as pd

# Reproducibility
rng = np.random.default_rng(42)

# ---------- Config ----------
N_ROWS = 20000
N_LEARNERS = 300
N_QUESTIONS = 600

TOPICS = ["History", "Geography", "Science", "Sports", "ArtsCulture", "Technology"]
DIFFICULTIES = [1, 2, 3]  # 1 easy, 2 medium, 3 hard

# ---------- Create learner profiles ----------
# Each learner has an underlying skill level (higher = better)
learner_ids = [f"L{str(i).zfill(4)}" for i in range(1, N_LEARNERS + 1)]
learner_skill = pd.DataFrame({
    "learner_id": learner_ids,
    "skill": rng.normal(loc=0.0, scale=1.0, size=N_LEARNERS)  # average around 0
})

# Some learners are more likely to use hints (help-seeking behaviour)
learner_skill["hint_tendency"] = rng.beta(a=2, b=5, size=N_LEARNERS)  # mostly low, some high

# ---------- Create question bank ----------
question_ids = [f"Q{str(i).zfill(4)}" for i in range(1, N_QUESTIONS + 1)]

q_topic = rng.choice(TOPICS, size=N_QUESTIONS, replace=True)
q_difficulty = rng.choice(DIFFICULTIES, size=N_QUESTIONS, replace=True, p=[0.45, 0.35, 0.20])  # more easy/medium than hard

questions = pd.DataFrame({
    "question_id": question_ids,
    "topic": q_topic,
    "difficulty": q_difficulty
})

# Topic difficulty bias: some topics slightly harder on average
topic_bias = {
    "History": 0.05,
    "Geography": 0.00,
    "Science": -0.10,       # slightly harder
    "Sports": 0.10,         # slightly easier
    "ArtsCulture": 0.00,
    "Technology": -0.05     # slightly harder
}

# ---------- Simulate interactions ----------
# Choose learner and question for each row
sim_learners = rng.choice(learner_ids, size=N_ROWS, replace=True)
sim_questions = rng.choice(question_ids, size=N_ROWS, replace=True)

df = pd.DataFrame({
    "learner_id": sim_learners,
    "question_id": sim_questions
})

# Merge question info
df = df.merge(questions, on="question_id", how="left")
df = df.merge(learner_skill, on="learner_id", how="left")

# ---------- Generate behaviour + outcome ----------
# Base probability of correctness depends on:
# learner skill (positive), difficulty (negative), topic bias, hints/attempts/time noise
difficulty_penalty = df["difficulty"].map({1: 0.2, 2: 0.7, 3: 1.2}).astype(float)
topic_effect = df["topic"].map(topic_bias).astype(float)

# Hint use probability increases with difficulty and learner hint tendency
p_hint = np.clip(0.08 + 0.10*(df["difficulty"]-1) + 0.35*df["hint_tendency"], 0, 0.85)
df["hints_used"] = rng.binomial(n=2, p=p_hint)  # 0-2 hints

# Attempts: more attempts on harder questions + lower skill
attempt_base = 1 + (df["difficulty"]-1) + (df["skill"] < 0).astype(int)
df["attempts_count"] = np.clip(attempt_base + rng.integers(0, 2, size=N_ROWS), 1, 5)

# Time spent: increases with difficulty, attempts, hints; decreases slightly with higher skill
time_mean = 25 + 18*(df["difficulty"]-1) + 10*(df["attempts_count"]-1) + 8*df["hints_used"] - 3*df["skill"]
time_noise = rng.normal(0, 8, size=N_ROWS)
df["time_spent_seconds"] = np.clip(time_mean + time_noise, 5, 240).round(0).astype(int)

# Correctness probability (logistic)
# Higher skill -> higher probability; harder difficulty -> lower; hints -> slightly help; many attempts -> signal struggle
logit = (
    0.6*df["skill"]
    - 0.9*difficulty_penalty
    + 0.4*topic_effect
    + 0.25*df["hints_used"]
    - 0.18*(df["attempts_count"]-1)
    - 0.004*(df["time_spent_seconds"]-30)
)

p_correct = 1 / (1 + np.exp(-logit))
df["correct"] = rng.binomial(n=1, p=np.clip(p_correct, 0.01, 0.99))

# Keep only the columns we want as "raw logs"
df_raw = df[[
    "learner_id", "question_id", "topic", "difficulty",
    "time_spent_seconds", "attempts_count", "hints_used", "correct"
]].copy()

df_raw.head()


,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct
0,L0202,Q0533,Sports,1,47,2,0,0
1,L0227,Q0249,History,1,37,2,0,1
2,L0115,Q0472,Sports,3,91,3,2,0
3,L0300,Q0506,Geography,2,67,2,1,0
4,L0150,Q0195,Sports,1,43,2,1,1


In [ ]:
df_raw.describe(include="all")


,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct
count,20000,20000,20000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
unique,300,600,6,NaN,NaN,NaN,NaN,NaN
top,L0103,Q0433,Science,NaN,NaN,NaN,NaN,NaN
freq,85,53,3706,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,1.724300,59.818150,2.753350,0.510350,0.326550
std,NaN,NaN,NaN,0.776994,26.264658,1.049414,0.626508,0.468963
min,NaN,NaN,NaN,1.000000,5.000000,1.000000,0.000000,0.000000
25%,NaN,NaN,NaN,1.000000,39.000000,2.000000,0.000000,0.000000
50%,NaN,NaN,NaN,2.000000,56.000000,3.000000,0.000000,0.000000
75%,NaN,NaN,NaN,2.000000,79.000000,3.000000,1.000000,1.000000


In [ ]:
print("Rows:", len(df_raw))
print("Unique learners:", df_raw["learner_id"].nunique())
print("Unique questions:", df_raw["question_id"].nunique())
print("Overall accuracy:", df_raw["correct"].mean().round(3))

print("\nAccuracy by difficulty:")
print(df_raw.groupby("difficulty")["correct"].mean().round(3))

print("\nAvg time by difficulty:")
print(df_raw.groupby("difficulty")["time_spent_seconds"].mean().round(1))


Rows: 20000
Unique learners: 300
Unique questions: 600
Overall accuracy: 0.327

Accuracy by difficulty:
difficulty
1    0.429
2    0.278
3    0.162
Name: correct, dtype: float64

Avg time by difficulty:
difficulty
1    38.5
2    68.0
3    97.2
Name: time_spent_seconds, dtype: float64


In [ ]:
df_raw.to_csv(RAW_FILE, index=False)
print("✅ Saved:", RAW_FILE)
print("File exists now?", os.path.exists(RAW_FILE))

✅ Saved: /content/drive/MyDrive/bigdata-gbl-quiz/data/raw/quiz_interactions.csv
File exists now? True


In [ ]:
df_check = pd.read_csv(RAW_FILE)
df_check.head(), df_check.shape


(  learner_id question_id      topic  difficulty  time_spent_seconds  \
 0      L0202       Q0533     Sports           1                  47   
 1      L0227       Q0249    History           1                  37   
 2      L0115       Q0472     Sports           3                  91   
 3      L0300       Q0506  Geography           2                  67   
 4      L0150       Q0195     Sports           1                  43   
 
    attempts_count  hints_used  correct  
 0               2           0        0  
 1               2           0        1  
 2               3           2        0  
 3               2           1        0  
 4               2           1        1  ,
 (20000, 8))